# Spaceship Titanic

**Competition:** [https://www.kaggle.com/competitions/spaceship-titanic](https://www.kaggle.com/competitions/spaceship-titanic)

**Task:** Predict which passengers were transported to an alternate dimension (binary classification).

**Metric:** Accuracy  &nbsp;|&nbsp;  **Public score achieved:** 0.799

## Approach
- Feature engineering: split `PassengerId` into group & group size, `Cabin` into deck/number/side, total spend & no-spend flag.
- LightGBM with native categorical handling, 5-fold stratified cross-validation, test predictions averaged across folds.

The cell below is a self-contained LightGBM pipeline. It reads the attached
competition data, prints the cross-validation score, and writes `submission.csv`.


In [ ]:
import warnings, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error
import lightgbm as lgb
warnings.filterwarnings("ignore")

def to_category(train, test, cols):
    for c in cols:
        train[c] = train[c].astype("category"); test[c] = test[c].astype("category")
        cats = pd.api.types.union_categoricals([train[c], test[c]]).categories
        train[c] = pd.Categorical(train[c], categories=cats)
        test[c] = pd.Categorical(test[c], categories=cats)
    return train, test

CLF = dict(n_estimators=2000, learning_rate=0.03, num_leaves=63, subsample=0.8,
           subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
           min_child_samples=30, random_state=42, n_jobs=-1, verbose=-1)
REG = dict(n_estimators=3000, learning_rate=0.02, num_leaves=31, subsample=0.8,
           subsample_freq=1, colsample_bytree=0.7, reg_lambda=1.0,
           min_child_samples=20, random_state=42, n_jobs=-1, verbose=-1)
CB = [lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)]

def cv_binary(X, y, Xt, proba=True, n=5):
    oof = np.zeros(len(X)); pred = np.zeros(len(Xt))
    for tr, va in StratifiedKFold(n, shuffle=True, random_state=42).split(X, y):
        m = lgb.LGBMClassifier(**CLF)
        m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[va], y.iloc[va])], callbacks=CB)
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]; pred += m.predict_proba(Xt)[:, 1] / n
    return oof, pred

def cv_multi(X, y, Xt, n=5):
    k = y.nunique(); oof = np.zeros((len(X), k)); pred = np.zeros((len(Xt), k))
    for tr, va in StratifiedKFold(n, shuffle=True, random_state=42).split(X, y):
        m = lgb.LGBMClassifier(**CLF)
        m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[va], y.iloc[va])], callbacks=CB)
        oof[va] = m.predict_proba(X.iloc[va]); pred += m.predict_proba(Xt) / n
    return oof, pred

def cv_reg(X, y, Xt, n=5):
    oof = np.zeros(len(X)); pred = np.zeros(len(Xt))
    for tr, va in KFold(n, shuffle=True, random_state=42).split(X):
        m = lgb.LGBMRegressor(**REG)
        m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[va], y.iloc[va])], callbacks=CB)
        oof[va] = m.predict(X.iloc[va]); pred += m.predict(Xt) / n
    return oof, pred

import glob, os
P = os.path.dirname(sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True))[0])
print("DATA DIR:", P)
tr = pd.read_csv(P + "/train.csv"); te = pd.read_csv(P + "/test.csv")
ids = te["PassengerId"]; y = tr["Transported"].astype(int)
def fe(df):
    df = df.copy()
    g = df["PassengerId"].str.split("_", expand=True)
    df["Group"] = g[0].astype(int); df["PersonNum"] = g[1].astype(int)
    df["GroupSize"] = df.groupby("Group")["PassengerId"].transform("count")
    cab = df["Cabin"].str.split("/", expand=True)
    df["Deck"] = cab[0]; df["CabinNum"] = pd.to_numeric(cab[1], errors="coerce"); df["Side"] = cab[2]
    spend = ["RoomService","FoodCourt","ShoppingMall","Spa","VRDeck"]
    df["TotalSpend"] = df[spend].sum(axis=1); df["NoSpend"] = (df["TotalSpend"] == 0).astype(int)
    for c in ["CryoSleep","VIP"]: df[c] = df[c].astype("float")
    return df.drop(columns=["PassengerId","Cabin","Name"])
X = fe(tr).drop(columns=["Transported"]); Xt = fe(te)
X, Xt = to_category(X, Xt, ["HomePlanet","Destination","Deck","Side"])
oof, pred = cv_binary(X, y, Xt)
print("CV accuracy:", accuracy_score(y, (oof > 0.5).astype(int)))
pd.DataFrame({"PassengerId": ids, "Transported": (pred > 0.5).astype(bool)}).to_csv("submission.csv", index=False)
